# vowl · Basic Tutorial

The main walkthrough of the **vowl** library for validating data against [ODCS](https://github.com/bitol-io/open-data-contract-standard) data contracts. Run it top-to-bottom.

This notebook covers the everyday workflow: setup, running a validation, and understanding the results. Three further notebooks build on it:

- `2_multiple_sources/`: validating one contract across multiple sources
- `3_real_databases/`: server-side validation with Testcontainers (needs Docker)
- `4_advanced_usage/`: explicit adapters (incl. `PooledAdapter`) and row filtering

## Contents

1. [Setup](#1-setup)
2. [Auto-Mapped Validation](#2-auto-mapped-validation): Pandas, Polars, other inputs
3. [The ValidationResult Object](#3-validationresult)
4. [Annotated Output: Your Full Table, Flagged](#4-annotated-output-your-full-table-flagged)

Generated CSV/JSON artifacts are written to this notebook's local `outputs/` folder.


<a id="1-setup"></a>
## 1. Setup

Install the dependencies (skipped during execution), then resolve the dataset paths used throughout the notebook.

In [1]:
# Install vowl (skipped during execution)
# %pip install 'vowl[all]'

In [2]:
from pathlib import Path

# Walk up to the repo root (works no matter how deep this notebook sits)
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "tests" / "hdb_resale").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# Single-source: HDB Resale dataset
HDB_DIR = REPO_ROOT / "tests" / "hdb_resale"
HDB_CSV = HDB_DIR / "HDBResaleWithErrors.csv"
HDB_CONTRACT = HDB_DIR / "hdb_resale_simple.yaml"  # Switch to "hdb_resale.yaml" for a more complex contract

# Multi-source: Employee dataset (payroll + employee list)
EMPLOYEE_DIR = REPO_ROOT / "tests" / "employee"
EMPLOYEE_PAYROLL_CSV = EMPLOYEE_DIR / "demo_employee_payroll.csv"
EMPLOYEE_LIST_CSV = EMPLOYEE_DIR / "demo_employee_list.csv"
EMPLOYEE_CONTRACT = EMPLOYEE_DIR / "employee_payroll_datacontract.yaml"

# Common imports used throughout this notebook
import pandas as pd
from vowl import validate_data

# --- Quieten known, benign warnings emitted by this demo ---
import warnings

# vowl surfaces UserWarnings (Arrow type coercion, multi-schema adapter reuse)
# that are informational for this demo dataset.
warnings.filterwarnings("ignore", category=UserWarning,
                        module=r"vowl\.validation\.runner")

In [3]:
# Confirm every dataset path resolves
print(f"HDB CSV exists:          {HDB_CSV.exists()}")
print(f"HDB contract exists:     {HDB_CONTRACT.exists()}")
print(f"Employee payroll CSV exists:  {EMPLOYEE_PAYROLL_CSV.exists()}")
print(f"Employee list CSV exists: {EMPLOYEE_LIST_CSV.exists()}")
print(f"Employee contract exists:     {EMPLOYEE_CONTRACT.exists()}")

HDB CSV exists:          True
HDB contract exists:     True
Employee payroll CSV exists:  True
Employee list CSV exists: True
Employee contract exists:     True


---

# Running a Validation

The simplest way to feed your data into vowl is to hand it a DataFrame and let it auto-detect the input type. For explicit adapters and row filtering, see the [Advanced Usage notebook](../4_advanced_usage/advanced_usage.ipynb).


<a id="2-auto-mapped-validation"></a>
## 2. Auto-Mapped Validation

When you pass a DataFrame directly to `validate_data(df=...)`, vowl uses a `DataSourceMapper` to automatically detect the input type (pandas, Polars, PyArrow, cuDF, Modin, etc.) and route it to the appropriate adapter -- in most cases, an in-memory DuckDB connection via `IbisAdapter`.

This is the simplest way to validate data. For explicit control over the adapter, see the [Advanced Usage notebook](../4_advanced_usage/advanced_usage.ipynb).

### 2.1 Pandas


In [4]:
import pandas as pd
from vowl import validate_data

df = pd.read_csv(HDB_CSV, low_memory=False).fillna("")

In [5]:
# Inspect the loaded data
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
df.head()

Loaded 201879 rows, 11 columns


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [6]:
result = validate_data(contract=str(HDB_CONTRACT), df=df)

In [7]:
# Show the full validation report
result.display_full_report()



=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       16 / 20 (80.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       16 / 20 (80.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       16 / 20 (80.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     201,861 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator      | expected      | actua

ValidationResult(passed=False, checks=20, passed_checks=16, failed_checks=4)

<a id="22-polars"></a>
### 2.2 Polars

vowl works with any [Narwhals-compatible](https://github.com/narwhals-dev/narwhals) DataFrame. Here we use Polars with the same contract.

In [8]:
import polars as pl

polars_df = pl.read_csv(HDB_CSV, infer_schema_length=9999999) # To allow mixed type data to be detected in this demo example
result = validate_data(contract=str(HDB_CONTRACT), df=polars_df)

In [9]:
print(f"Loaded {len(polars_df)} rows via Polars")
result.display_full_report()

Loaded 201879 rows via Polars


=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       17 / 20 (85.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       17 / 20 (85.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     201,863 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operato

ValidationResult(passed=False, checks=20, passed_checks=17, failed_checks=3)

#### Why Do Pandas and Polars Show Different Failure Counts?

Pandas reports **4 failed checks** (16/20 pass) while Polars reports **3 failed checks** (17/20 pass). The differences stem from how each library handles null/empty values before they reach DuckDB:

| Check | Pandas | Polars | Reason |
|-------|--------|--------|--------|
| **AddressBlockHouseNumber** | FAILED (1 row) | PASSED | Pandas `fillna("")` converts a null `block` to an empty string `""`, which fails the regex `^[A-Za-z0-9]{1,10}$`. Polars keeps it as `null`, and SQL `WHERE ... !~ pattern` evaluates to `NULL` (not `TRUE`), so the row is not counted. |
| **Year** | FAILED (3 rows) | FAILED (2 rows) | One record has an empty `lease_commence_date`. In Pandas, `fillna("")` turns it into `""` which fails the `^[0-9]{4}$` regex. In Polars, the value stays `null` and is skipped by the WHERE clause. |

This is a documented aspect of null handling across backends -- see [Known Issues: Null Handling](../../docs/known-issues.md#null-handling-varies-across-backends). If you need to catch nulls explicitly, add a `nullValues` library check to your contract.

### 2.3 Other Inputs (Connection Strings, Ibis Backends, Spark, Arrow)

The auto-mapper accepts more than DataFrames. vowl detects the type and creates the appropriate `IbisAdapter` automatically.

In [10]:
# The auto-mapper supports more than just DataFrames.
# Below are other input types that validate_data() can auto-detect:

# --- Connection string (any URI that Ibis recognises) ---
# result = validate_data(contract=str(HDB_CONTRACT), df="postgresql://<user>:<password>@<host>:<port>/<dbname>")
# result = validate_data(contract=str(HDB_CONTRACT), df="snowflake://<user>:<password>@<account>/<db>/<schema>")

# --- Ibis backend (already connected) ---
# con = ibis.duckdb.connect("my_database.ddb")
# result = validate_data(contract=str(HDB_CONTRACT), df=con)

# --- PySpark DataFrame ---
# spark_df = spark.read.table("my_table")
# result = validate_data(contract=str(HDB_CONTRACT), df=spark_df)

# --- PyArrow Table ---
# import pyarrow.csv as pcsv
# arrow_table = pcsv.read_csv("data.csv")
# result = validate_data(contract=str(HDB_CONTRACT), df=arrow_table)

---

# Understanding the Results

Every `validate_data(...)` call returns a `ValidationResult`. This section covers the accessors it exposes -- summary DataFrames, failed-row views, and the annotated full table.

<a id="3-validationresult"></a>
## 3. The ValidationResult Object

The `ValidationResult` object provides several ways to inspect and export results. First, run a validation to work with.


In [11]:
# Run a validation to work with
result = validate_data(contract=str(HDB_CONTRACT), df=pd.read_csv(HDB_CSV, low_memory=False).fillna(""))

In [12]:
# Check overall pass/fail, then print the summary (without failed rows)
print(f"All checks passed: {result.passed}")
result.print_summary()

All checks passed: False


=== Data Quality Validation Results ===
   Contract Version:      v3.1.0
   Contract ID:           c11443ee-542f-4442-b28d-2d224342be37
   Schemas:               hdb_resale_prices

 OVERALL DATA QUALITY
   Overall:
     Checks Pass Rate:       16 / 20 (80.0%)

   hdb_resale_prices:
     Overall:
       Checks Pass Rate:       16 / 20 (80.0%)
       ERRORED Checks:         0
     Single Table:
       Checks Pass Rate:       16 / 20 (80.0%)
       ERRORED Checks:         0
       Unique Passed Rows:     201,861 / 201,879 (99.9%)
     Multi Table:
       Checks Pass Rate:       0 / 0 (N/A)
       ERRORED Checks:         0
       Non-unique Failed Rows: 0


 CHECK RESULTS
+-----------------------------------------+---------------------------------------+-------------------+--------+---------------+---------------+--------+----------------+
| check_id                                | Target                                | tables_in_query   | status | operator    

ValidationResult(passed=False, checks=20, passed_checks=16, failed_checks=4)

### Check Results DataFrame

`get_check_results_df()` returns a single DataFrame with one row per check: status, expected/actual values, execution time, dimension, and more.

In [13]:
# Full check results as a DataFrame (one row per check)
check_results_df = result.get_check_results_df()

# To include contract and check definitions in the DataFrame, use:
# check_results_df = result.get_check_results_df(include_contract_definition=True, include_check_definition=True)

In [14]:
# View the check results
check_results_df.to_pandas()

,check_name,target,schema_name,engine,status,operator,actual_value,expected_value,failed_rows_count,aggregation_type,message,rendered_implementation,tables_in_query,check_path,check_ref_type,logical_type,is_generated,execution_time_ms
0,month_column_exists_check,hdb_resale_prices.month,hdb_resale_prices,sql,PASSED,mustBe,0,0,0,count,,"SELECT COUNT(*) FROM (SELECT ""month"" FROM ""hdb...",['hdb_resale_prices'],$.schema[0].properties[0].name,DeclaredColumnExistsCheckReference,string,True,4.385166
1,month_logical_type_check,hdb_resale_prices.month,hdb_resale_prices,sql,PASSED,mustBe,0,0,0,count,,"SELECT COUNT(*) FROM ""hdb_resale_prices"" WHERE...",['hdb_resale_prices'],$.schema[0].properties[0].logicalType,LogicalTypeCheckReference,string,True,7.890958
2,town_column_exists_check,hdb_resale_prices.town,hdb_resale_prices,sql,PASSED,mustBe,0,0,0,count,,"SELECT COUNT(*) FROM (SELECT ""town"" FROM ""hdb_...",['hdb_resale_prices'],$.schema[0].properties[1].name,DeclaredColumnExistsCheckReference,NaN,True,3.833125
3,flat_type_column_exists_check,hdb_resale_prices.flat_type,hdb_resale_prices,sql,PASSED,mustBe,0,0,0,count,,"SELECT COUNT(*) FROM (SELECT ""flat_type"" FROM ...",['hdb_resale_prices'],$.schema[0].properties[2].name,DeclaredColumnExistsCheckReference,NaN,True,3.973250
4,block_column_exists_check,hdb_resale_prices.block,hdb_resale_prices,sql,PASSED,mustBe,0,0,0,count,,"SELECT COUNT(*) FROM (SELECT ""block"" FROM ""hdb...",['hdb_resale_prices'],$.schema[0].properties[3].name,DeclaredColumnExistsCheckReference,NaN,True,4.069708
5,street_name_column_exists_check,hdb_resale_prices.street_name,hdb_resale_prices,sql,PASSED,mustBe,0,0,0,count,,"SELECT COUNT(*) FROM (SELECT ""street_name"" FRO...",['hdb_resale_prices'],$.schema[0].properties[4].name,DeclaredColumnExistsCheckReference,NaN,True,4.103875
6,storey_range_column_exists_check,hdb_resale_prices.storey_range,hdb_resale_prices,sql,PASSED,mustBe,0,0,0,count,,"SELECT COUNT(*) FROM (SELECT ""storey_range"" FR...",['hdb_resale_prices'],$.schema[0].properties[5].name,DeclaredColumnExistsCheckReference,NaN,True,4.113542
7,floor_area_sqm_column_exists_check,hdb_resale_prices.floor_area_sqm,hdb_resale_prices,sql,PASSED,mustBe,0,0,0,count,,"SELECT COUNT(*) FROM (SELECT ""floor_area_sqm"" ...",['hdb_resale_prices'],$.schema[0].properties[6].name,DeclaredColumnExistsCheckReference,NaN,True,4.819709
8,flat_model_column_exists_check,hdb_resale_prices.flat_model,hdb_resale_prices,sql,PASSED,mustBe,0,0,0,count,,"SELECT COUNT(*) FROM (SELECT ""flat_model"" FROM...",['hdb_resale_prices'],$.schema[0].properties[7].name,DeclaredColumnExistsCheckReference,NaN,True,4.551542
9,lease_commence_date_column_exists_check,hdb_resale_prices.lease_commence_date,hdb_resale_prices,sql,PASSED,mustBe,0,0,0,count,,"SELECT COUNT(*) FROM (SELECT ""lease_commence_d...",['hdb_resale_prices'],$.schema[0].properties[8].name,DeclaredColumnExistsCheckReference,NaN,True,4.337208


### Failed Rows (per check)

`get_output_dfs()` returns a dict of `{check_id: DataFrame}`, one entry per failed check, containing the actual rows that failed.

In [15]:
# Failed rows grouped by check. Each key is "schema::check_name"
output_dfs = result.get_output_dfs()

In [16]:
# Display the failed rows for each check
for check_id, failed_df in output_dfs.items():
    if len(failed_df) == 0:
        continue  # Skip checks with no failed rows
    print(f"\n{'='*60}")
    print(f"Check: {check_id}  ({len(failed_df)} failed rows)")
    print(f"{'='*60}")
    display(failed_df.to_pandas().head())


Check: hdb_resale_prices::AddressBlockHouseNumber  (1 failed rows)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,check_id,tables_in_query
0,2017-01,BEDOK,4 ROOM,,BEDOK RESERVOIR RD,07 TO 09,84.0,Simplified,1986,68 years 10 months,395000.0,AddressBlockHouseNumber,hdb_resale_prices



Check: hdb_resale_prices::Month  (2 failed rows)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,check_id,tables_in_query
0,2017-jan,BEDOK,5 ROOM,21,CHAI CHEE RD,07 TO 09,130.0,Adjoined flat,1972,54 years 06 months,530000.0,Month,hdb_resale_prices
1,2017-jan,BISHAN,3 ROOM,105,BISHAN ST 12,04 TO 06,4.0,Simplified,1985,67 years 11 months,395000.0,Month,hdb_resale_prices



Check: hdb_resale_prices::Year  (3 failed rows)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,check_id,tables_in_query
0,2017-01,ANG MO KIO,3 ROOM,219,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1977.0,59 years 06 months,297000.0,Year,hdb_resale_prices
1,2017-01,ANG MO KIO,3 ROOM,211,ANG MO KIO AVE 3,01 TO 03,67.0,New Generation,abc,59 years 03 months,325000.0,Year,hdb_resale_prices
2,2017-01,ANG MO KIO,3 ROOM,330,ANG MO KIO AVE 1,07 TO 09,68.0,New Generation,,63 years,338000.0,Year,hdb_resale_prices



Check: hdb_resale_prices::floor_area_must_be_less_than_200  (12 failed rows)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,check_id,tables_in_query
0,2017-06,KALLANG/WHAMPOA,3 ROOM,38,JLN BAHAGIA,01 TO 03,215.0,Terrace,1972,54 years 01 month,830000.0,floor_area_must_be_less_than_200,hdb_resale_prices
1,2017-09,CHOA CHU KANG,EXECUTIVE,641,CHOA CHU KANG ST 64,16 TO 18,215.0,Premium Maisonette,1998,79 years 04 months,888000.0,floor_area_must_be_less_than_200,hdb_resale_prices
2,2017-12,KALLANG/WHAMPOA,3 ROOM,65,JLN MA'MOR,01 TO 03,249.0,Terrace,1972,53 years 07 months,1053888.0,floor_area_must_be_less_than_200,hdb_resale_prices
3,2018-01,CHOA CHU KANG,EXECUTIVE,639,CHOA CHU KANG ST 64,10 TO 12,215.0,Premium Maisonette,1998,79 years,900000.0,floor_area_must_be_less_than_200,hdb_resale_prices
4,2018-09,KALLANG/WHAMPOA,3 ROOM,41,JLN BAHAGIA,01 TO 03,237.0,Terrace,1972,52 years 10 months,1185000.0,floor_area_must_be_less_than_200,hdb_resale_prices


### Consolidated Failed Rows (per table)

`get_consolidated_output_dfs()` deduplicates failed rows across all checks and groups them by table. Useful when multiple checks flag the same row.

In [17]:
# Consolidated: deduplicated failed rows grouped by table
consolidated = result.get_consolidated_output_dfs()

/var/folders/tg/0hbw1fbs5q1_5kdtp36klrtw0000gn/T/ipykernel_4359/3978688020.py:2: DeprecationWarning: get_consolidated_output_dfs() is deprecated and will be removed in a future release. Use get_annotated_output() instead, which returns your full tables with failing rows flagged in place plus per-check residues. Note the output shape differs: annotated tables carry a 'check_info' JSON-array column (per row) rather than the grouped 'check_ids' comma-joined string, and include passing rows too.
  consolidated = result.get_consolidated_output_dfs()


In [18]:
# Display the deduplicated failed rows per table
for table_name, consolidated_df in consolidated.items():
    print(f"\n{'='*60}")
    print(f"Table: {table_name}  ({len(consolidated_df)} unique failed rows)")
    print(f"{'='*60}")
    display(consolidated_df.to_pandas().head())


Table: hdb_resale_prices  (18 unique failed rows)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,check_ids,tables_in_query
0,2017-01,BEDOK,4 ROOM,,BEDOK RESERVOIR RD,07 TO 09,84.0,Simplified,1986,68 years 10 months,395000.0,AddressBlockHouseNumber,hdb_resale_prices
1,2017-jan,BEDOK,5 ROOM,21,CHAI CHEE RD,07 TO 09,130.0,Adjoined flat,1972,54 years 06 months,530000.0,Month,hdb_resale_prices
2,2017-jan,BISHAN,3 ROOM,105,BISHAN ST 12,04 TO 06,4.0,Simplified,1985,67 years 11 months,395000.0,Month,hdb_resale_prices
3,2017-01,ANG MO KIO,3 ROOM,219,ANG MO KIO AVE 1,07 TO 09,67.0,New Generation,1977.0,59 years 06 months,297000.0,Year,hdb_resale_prices
4,2017-01,ANG MO KIO,3 ROOM,211,ANG MO KIO AVE 3,01 TO 03,67.0,New Generation,abc,59 years 03 months,325000.0,Year,hdb_resale_prices


### Saving to Disk

`result.save(...)` writes the results (CSVs + a JSON summary) to a directory.

In [19]:
# Save results to disk (CSV + JSON summary)
result.save(output_dir="outputs", prefix="vowl_demo_HDB_results")


Results saved:
   - outputs/vowl_demo_HDB_results_check_results.csv
   - outputs/vowl_demo_HDB_results_hdb_resale_prices.csv
   - outputs/vowl_demo_HDB_results_summary.json


/var/folders/tg/0hbw1fbs5q1_5kdtp36klrtw0000gn/T/ipykernel_4359/3524666264.py:2: DeprecationWarning: save() currently defaults to output_mode='failed_rows', which writes the deprecated consolidated failed-rows CSVs (grouped rows with a 'check_ids' column). This default will change to 'annotated' in a future release. Pass output_mode='annotated' now to opt in early (full tables with a per-row 'check_info' column plus per-check residues), or output_mode='failed_rows' to keep the old shape explicitly and silence this warning.
  result.save(output_dir="outputs", prefix="vowl_demo_HDB_results")


ValidationResult(passed=False, checks=20, passed_checks=16, failed_checks=4)

<a id="4-annotated-output"></a>
## 4. Annotated Output: Your Full Table, Flagged

The main accessor here is **`get_annotated_output()`**. It returns your full table with one extra column, `check_info`, so you can see failures in the context of every row instead of on their own.

It returns a dict with two keys:

- **`"annotated"`**: `{schema: table}`. Each table is your full data plus a `check_info` column. Every original row is kept. `check_info` is `null` for rows that passed every check, and a list of the failing check(s) for rows that didn't.
- **`"residues"`**: failed rows for checks that can't be attached to a single table (cross-table, aggregation, and column-subset checks). The single-table HDB contract below has none; the [residual-rows example](#residual-non-mergeable-rows) at the end of this section shows them using a multi-source contract.

> Not every check can be merged into the annotated table. For the full rules and worked examples, see [Known Issues: Annotated Output](../../docs/known-issues.md#annotated-output-not-all-checks-can-be-merged).


### Choosing How Much Detail: the `check_info` Options

`check_info` is a **list of objects, one per failing check** (stored as a JSON string in the column). The `check_info=` argument sets how many fields each object carries:

| `check_info` | Each object contains | Use it when |
|---|---|---|
| `"names"` *(default)* | `check_name` | You only need to know *which* checks a row failed |
| `"summary"` | `check_name`, `dimension`, `tags`, `target` | You want each failure's quality dimension, tags, and column (e.g. to split completeness vs. accuracy) |
| `"full"` | the entire check definition (`type`, `description`, `query`, `mustBe`, `dimension`, `tags`) plus `check_name` and `target` | You want everything, including the underlying rule/SQL, on every row |

All three return the **same shape** (a list of objects) so you always read a value the same way (`json.loads(cell)[0]["check_name"]`); they differ only in how many fields each object has. A row failing two checks gets a two-object list.

```text
"names"    [{"check_name": "Month"}]
"summary"  [{"check_name": "Month", "dimension": "conformity",
             "tags": ["SG-DRM v5.0"], "target": "hdb_resale_prices.month"}]
"full"     [{"type": "sql", "name": "Month", "description": "...", "mustBe": 0,
             "query": "SELECT COUNT(*) ...", "tags": ["SG-DRM v5.0"],
             "dimension": "conformity", "check_name": "Month",
             "target": "hdb_resale_prices.month"}]
```

The examples below use `"summary"`.

In [20]:
# "annotated" -> {schema: your full table + a check_info column}
# "residues"  -> failed rows for checks that can't attach to one table (shown later)
# check_info="summary" -> each object also carries dimension, tags, and target.
output = result.get_annotated_output(check_info="summary")
annotated = output["annotated"]["hdb_resale_prices"].to_pandas()

In [21]:
# Inspect the structure of the annotated output
print("Top-level keys:    ", list(output.keys()))
print("Annotated schemas: ", list(output["annotated"].keys()))
print("Residue keys:      ", list(output["residues"].keys()))
print(f"\nAnnotated table: {annotated.shape[0]:,} rows x {annotated.shape[1]} columns")
print("Columns:", list(annotated.columns))

Top-level keys:     ['annotated', 'residues']
Annotated schemas:  ['hdb_resale_prices']
Residue keys:       []

Annotated table: 201,879 rows x 12 columns
Columns: ['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'check_info']


### Inspecting the Full Table, Flagged Rows First

Every original row is present. `check_info` is `null` for rows that passed and a JSON list for rows that failed, one object per failing check (here carrying `dimension`, `tags`, and `target` from the `"summary"` preset). Sorting by `check_info` (nulls last) floats the flagged rows to the top while still showing the whole table.

In [22]:
# Sort the full table so flagged rows surface first; passing rows (check_info = null) sink.
flagged_first = annotated.sort_values("check_info", na_position="last").reset_index(drop=True)
n_flagged = annotated["check_info"].notna().sum()

In [23]:
print(f"{n_flagged} flagged rows at the top, {len(annotated) - n_flagged:,} passing rows below\n")
flagged_first[["month", "town", "block", "floor_area_sqm",
               "lease_commence_date", "check_info"]].head(20)

18 flagged rows at the top, 201,861 passing rows below



,month,town,block,floor_area_sqm,lease_commence_date,check_info
0,2017-01,BEDOK,,84.0,1986,"[{""check_name"": ""AddressBlockHouseNumber"", ""di..."
1,2017-jan,BEDOK,21,130.0,1972,"[{""check_name"": ""Month"", ""dimension"": ""conform..."
2,2017-jan,BISHAN,105,4.0,1985,"[{""check_name"": ""Month"", ""dimension"": ""conform..."
3,2017-01,ANG MO KIO,219,67.0,1977.0,"[{""check_name"": ""Year"", ""dimension"": ""conformi..."
4,2017-01,ANG MO KIO,211,67.0,abc,"[{""check_name"": ""Year"", ""dimension"": ""conformi..."
5,2017-01,ANG MO KIO,330,68.0,,"[{""check_name"": ""Year"", ""dimension"": ""conformi..."
6,2017-06,KALLANG/WHAMPOA,38,215.0,1972,"[{""check_name"": ""floor_area_must_be_less_than_..."
7,2017-09,CHOA CHU KANG,641,215.0,1998,"[{""check_name"": ""floor_area_must_be_less_than_..."
8,2017-12,KALLANG/WHAMPOA,65,249.0,1972,"[{""check_name"": ""floor_area_must_be_less_than_..."
9,2018-01,CHOA CHU KANG,639,215.0,1998,"[{""check_name"": ""floor_area_must_be_less_than_..."


### Keeping the Clean Rows for Downstream Use

Because the annotation lives on the full table, separating the good rows from the bad is a one-liner. Filter to where `check_info` is null, drop the annotation column, and you have a clean dataset ready to feed downstream -- no separate join back to the source needed.

In [24]:
# Keep only the rows that passed every check, then drop the annotation column
clean = annotated[annotated["check_info"].isna()].drop(columns=["check_info"])

In [25]:
print(f"Clean rows ready for downstream use: {len(clean):,} of {len(annotated):,}")
clean.head()

Clean rows ready for downstream use: 201,861 of 201,879


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


<a id="residual-rows"></a>
### Residual (Non-Mergeable) Rows

The HDB examples above are single-table, so `residues` is empty. Residues appear when a check can't be attached to one table: **aggregation** checks, **column-subset** checks, and **cross-table** checks whose failed rows carry columns from more than the anchor table.

Residues are **per-check**: `get_annotated_output()` returns one entry for each non-mergeable check, keyed `"<schema>::<check_name>"`. They are never grouped together, so a row that fails two such checks appears once under each check's entry. Each entry carries the failed rows plus the same `check_info` column the annotated tables use (a single-element JSON array, shaped by the `check_info` preset) and `tables_in_query`, so everything `get_annotated_output()` returns is read the same way.

> **A cross-table check can *merge* instead of becoming a residue.** If you shape its failed-rows query to project only the anchor table's columns (e.g. `SELECT payroll.*` inside a subquery), the orphan rows match that schema and land directly in its `check_info` column, with no residue. The Employee contract below carries both shapes: `orphan_payroll_rows_merge_onto_payroll` (subquery-projected, merges onto `demo_employee_payroll`) and `employee_id_exists_in_master_list` / `phone_number_exists_in_master_list` (bare JOINs, stay residues). See [Known Issues: Annotated Output](../../docs/known-issues.md#annotated-output-not-all-checks-can-be-merged) for the rules.

Below is a quick multi-source run on the Employee dataset, whose contract has cross-table checks. (Multi-source validation is covered in the [Multiple Sources notebook](../2_multiple_sources/multiple_sources.ipynb); we borrow it here just to produce residues.)

In [26]:
import ibis
from vowl.adapters import IbisAdapter

# Load the two Employee tables onto one DuckDB connection (cross-table contract)
con = ibis.duckdb.connect()
con.create_table("demo_employee_payroll", pd.read_csv(EMPLOYEE_PAYROLL_CSV))
con.create_table("demo_employee_list", pd.read_csv(EMPLOYEE_LIST_CSV))

mt_result = validate_data(contract=str(EMPLOYEE_CONTRACT), adapter=IbisAdapter(con))
mt_output = mt_result.get_annotated_output()

In [27]:
# Each residue is the failed rows for ONE non-mergeable check, keyed "<schema>::<check>"
print("Annotated schemas:", list(mt_output["annotated"].keys()))
print("Residue keys:     ", list(mt_output["residues"].keys()))

for key, residue in mt_output["residues"].items():
    residue_df = residue.to_pandas()
    print(f"\nResidue '{key}': {len(residue_df)} failed row(s)")
    display(residue_df[["employee_id", "payroll_id", "month",
                        "check_info", "tables_in_query"]])

Annotated schemas: ['demo_employee_payroll', 'demo_employee_list']
Residue keys:      ['demo_employee_payroll::employee_id_exists_in_master_list', 'demo_employee_payroll::phone_number_exists_in_master_list']

Residue 'demo_employee_payroll::employee_id_exists_in_master_list': 1 failed row(s)


,employee_id,payroll_id,month,check_info,tables_in_query
0,e939123,e52e556f-79b0-471f-ad08-e27b2c524ace,2025-12,"[{""check_name"": ""employee_id_exists_in_master_...","demo_employee_list, demo_employee_payroll"



Residue 'demo_employee_payroll::phone_number_exists_in_master_list': 2 failed row(s)


,employee_id,payroll_id,month,check_info,tables_in_query
0,e128903,cb04c5bb-9386-44cf-a565-2276744c9cc0,2025-12,"[{""check_name"": ""phone_number_exists_in_master...","demo_employee_list, demo_employee_payroll"
1,e939123,e52e556f-79b0-471f-ad08-e27b2c524ace,2025-12,"[{""check_name"": ""phone_number_exists_in_master...","demo_employee_list, demo_employee_payroll"


#### The Other Side: a Cross-Table Check That *Merges*

Whether a cross-table check merges is decided entirely by **the columns its failed rows come back with**: they have to match the anchor table's columns exactly. The two checks above join `payroll` against a reference table with a bare `SELECT *`, so their failed rows carry columns from *both* tables and can't land on any single table, so they become a residue.

The *same* referential question **merges** when you wrap it so the inner query projects only the payroll columns (`SELECT payroll.* ...`). The contract's `orphan_payroll_rows_merge_onto_payroll` check does exactly that, so its failed rows come back with exactly the payroll columns and are annotated **directly onto the payroll table's `check_info` column** (notice it's absent from the residue keys above).

> For the full mechanics (how vowl derives the scalar and failed-rows queries, and why the `COUNT(*)` -> `SELECT *` rewrite only touches the outer projection), see [Known Issues: Cross-table checks, mergeable when the failed rows match the home schema](../../docs/known-issues.md#1-cross-table-checks-mergeable-when-the-failed-rows-match-the-home-schema).


In [28]:
# The subquery-projected cross-table check lands on the payroll annotated table,
# not in residues. Find the payroll rows it flagged.
payroll_annotated = mt_output["annotated"]["demo_employee_payroll"].to_pandas()

MERGED_CHECK = "orphan_payroll_rows_merge_onto_payroll"
flagged = payroll_annotated[
    payroll_annotated["check_info"].fillna("").str.contains(MERGED_CHECK)
]

print(f"'{MERGED_CHECK}' in residues? "
      f"{any(MERGED_CHECK in k for k in mt_output['residues'])}")
print(f"Payroll rows it annotated: {len(flagged)}\n")
display(flagged[["employee_id", "month", "check_info"]])

'orphan_payroll_rows_merge_onto_payroll' in residues? False
Payroll rows it annotated: 1



,employee_id,month,check_info
2,e939123,2025-12,"[{""check_name"": ""phone_number_logical_type_opt..."


### Writing Annotated Tables to Disk

`result.save(...)` can write the annotated tables instead of (or alongside) the failed-rows CSVs. The `output_mode` argument controls the layout (every mode also writes `<prefix>_check_results.csv` and `<prefix>_summary.json`):

| Mode | Files written |
|------|----------------|
| `"failed_rows"` (default) | One grouped failed-rows CSV per table key, e.g. `<prefix>_<table>.csv` |
| `"annotated"` | One `<prefix>_<schema>_annotated.csv` per schema (full table + `check_info`), **plus** one `<prefix>_<schema>_<check>_residue.csv` per non-mergeable check |
| `"both"` | The failed-rows CSVs **and** the annotated CSVs (residues are already covered by the grouped failed-rows CSVs, so no separate `_residue` files are written) |

The `check_info` argument (`"names"` / `"summary"` / `"full"`, described above) sets how much detail the saved `check_info` column carries. To make annotated output the default for every `save()`, pass `ValidationConfig(output_mode="annotated", annotated_check_info="summary")` to `validate_data(config=...)`.

Single-table contracts (like the HDB example) have no non-mergeable checks, so they never produce `_residue` files. The multi-source run below does, so let's save it to see them.

In [29]:
# Write the annotated table(s) to disk
result.save(output_dir="outputs", prefix="vowl_demo_annotated", output_mode="annotated")


Results saved:
   - outputs/vowl_demo_annotated_check_results.csv
   - outputs/vowl_demo_annotated_hdb_resale_prices_annotated.csv
   - outputs/vowl_demo_annotated_summary.json


ValidationResult(passed=False, checks=20, passed_checks=16, failed_checks=4)

#### Saving a Run That Has Residues

`result` above is single-table, so its `save(output_mode="annotated")` wrote only `*_annotated.csv` files. Saving the multi-source `mt_result` from the residue example instead also writes one `*_residue.csv` per non-mergeable check, keyed `<schema>_<check>` (the `::` in the residue key becomes `_`). Each residue CSV carries that one check's failed rows plus the same `check_info` column as the annotated tables and `tables_in_query`.

In [30]:
import os

# Save the multi-source run; annotated mode emits residue CSVs for the
# cross-table checks that can't be merged onto a single table.
mt_result.save(output_dir="outputs", prefix="vowl_demo_residues", output_mode="annotated")

print("\nFiles written:")
for fname in sorted(f for f in os.listdir("outputs") if f.startswith("vowl_demo_residues")):
    tag = "  <- residue" if fname.endswith("_residue.csv") else ""
    print(f"   {fname}{tag}")


Results saved:
   - outputs/vowl_demo_residues_check_results.csv
   - outputs/vowl_demo_residues_demo_employee_payroll_annotated.csv
   - outputs/vowl_demo_residues_demo_employee_list_annotated.csv
   - outputs/vowl_demo_residues_demo_employee_payroll_employee_id_exists_in_master_list_residue.csv
   - outputs/vowl_demo_residues_demo_employee_payroll_phone_number_exists_in_master_list_residue.csv
   - outputs/vowl_demo_residues_summary.json

Files written:
   vowl_demo_residues_check_results.csv
   vowl_demo_residues_demo_employee_list_annotated.csv
   vowl_demo_residues_demo_employee_payroll_annotated.csv
   vowl_demo_residues_demo_employee_payroll_employee_id_exists_in_master_list_residue.csv  <- residue
   vowl_demo_residues_demo_employee_payroll_phone_number_exists_in_master_list_residue.csv  <- residue
   vowl_demo_residues_summary.json


### Where the Full Check Definitions Live

To keep the annotated table readable, you'll usually pick `"names"` or `"summary"` rather than `"full"`. You don't lose anything: the complete definition of **every** check (its `dimension`, `tags`, `description`, `query`, and more) is always written to the `*_summary.json` file that `save()` produces, keyed by check name. Keep the table light, and look up the rest here when you need it.

In [31]:
import json

# save(prefix="vowl_demo_annotated", ...) wrote this alongside the CSVs.
with open("outputs/vowl_demo_annotated_summary.json") as f:
    summary = json.load(f)

# Every check's resolved definition lives under "check_results", keyed by name.
month_check = next(c for c in summary["check_results"] if c["name"] == "Month")

print("Target:    ", month_check["target"])
print("Dimension: ", month_check["check_definition"]["dimension"])
print("Tags:      ", month_check["check_definition"]["tags"])
print("\nFull check_definition:")
print(json.dumps(month_check["check_definition"], indent=2))

Target:     hdb_resale_prices.month
Dimension:  conformity
Tags:       ['SG-DRM v5.0']

Full check_definition:
{
  "type": "sql",
  "name": "Month",
  "description": "Based on ISO 8601, assumed to be in UTC +8 | YYYY-MM",
  "mustBe": 0,
  "query": "SELECT COUNT(*)\nFROM \"hdb_resale_prices\"\nWHERE CAST(month AS TEXT) !~ '^[0-9]{4}-(0[1-9]|1[0-2])$';",
  "tags": [
    "SG-DRM v5.0"
  ],
  "dimension": "conformity"
}
